# LocationRecovery

> **Created by Codex.**

Build generic location-recovery graphs from pairwise direction measurements.

Source: [`LocationRecovery.h`](https://github.com/borglab/gtsam/blob/develop/gtsam/sfm/LocationRecovery.h)

GTSAM Copyright 2010-2026, Georgia Tech Research Corporation,
Atlanta, Georgia 30332-0415
All Rights Reserved

Authors: Frank Dellaert, et al. (see THANKS for the full author list)

See LICENSE for the license information

<a href="https://colab.research.google.com/github/borglab/gtsam/blob/develop/gtsam/sfm/doc/LocationRecovery.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
try:
    import google.colab
    %pip install --quiet gtsam-develop
except ImportError:
    pass

In [ ]:
import gtsam
import numpy as np

from gtsam import symbol_shorthand

C = symbol_shorthand.C
K = symbol_shorthand.K
P = symbol_shorthand.P
S = symbol_shorthand.S
X = symbol_shorthand.X

## Role

`LocationRecovery` is the unopinionated base for direction-based position estimation. Edges may connect any `Point3` variables. `bilinear=False` creates chordal `TranslationFactor`s; `bilinear=True` creates BATA factors plus one scale variable per edge.

The class builds and initializes graphs but deliberately does not choose a complete gauge beyond the anchor helper.

In [ ]:
direction_noise = gtsam.noiseModel.Isotropic.Sigma(2, 0.02)
edges = [
    gtsam.BinaryMeasurementUnit3(X(0), X(1), gtsam.Unit3(np.array([1.0, 0.0, 0.0])), direction_noise),
    gtsam.BinaryMeasurementUnit3(X(1), X(2), gtsam.Unit3(np.array([0.0, 1.0, 0.0])), direction_noise),
]

recovery = gtsam.LocationRecovery()
graph = recovery.buildGraph(edges, bilinear=True)
recovery.addAnchorPrior(X(0), graph)
initial = recovery.initializeRandomly({X(0), X(1), X(2)}, len(edges), True)

print("graph factors:", graph.size())
print("initial keys:", initial.keys())

## Gauge warning

An anchor removes global translation but direction-only geometry also has a scale gauge. Add a physically meaningful distance, position, or scale constraint before optimizing. `TranslationRecovery` supplies a two-key gauge policy; `GlobalPositioner` supplies a camera/landmark-specific workflow.